# Compilation du GeoPackage final

Ce notebook vérifie que `database.geopackage.compile_geopackage` rassemble toutes les couches traitées (`data/processed/vector/`) en un unique GeoPackage multi-couches (`data/processed/database/projet.gpkg`), prêt à ouvrir dans QGIS ou ArcGIS Pro.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from database.geopackage import compile_geopackage

## 1. Compilation

In [ ]:
processed_vector_dir = repo_root / "data" / "processed" / "vector"
output_path = repo_root / "data" / "processed" / "database" / "projet.gpkg"

compile_geopackage(processed_vector_dir, output_path)
print("Taille :", round(output_path.stat().st_size / (1024 * 1024), 2), "Mo")

## 2. Couches présentes dans le GeoPackage

In [ ]:
import fiona

for layer in fiona.listlayers(output_path):
    with fiona.open(output_path, layer=layer) as src:
        print(f"{layer:45s} {len(src):>6} entites   {src.schema['geometry']:15s} {src.crs}")

## 3. Aperçu sur une carte

Quelques couches représentatives, relues directement depuis le GeoPackage compilé. Backend `folium`, plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import geopandas as gpd
import leafmap.foliumap as leafmap

preview_layers = {
    "bd_topo_batiment": {"color": "orange", "fillOpacity": 0.5},
    "bd_topo_troncon_de_route": {"color": "gray", "weight": 2},
    "cadastre_parcelles_38348": {"color": "black", "weight": 1, "fillOpacity": 0.05},
    "adresses_38348": {"color": "blue", "fillOpacity": 0.6, "radius": 2},
}

m = leafmap.Map()

last_gdf_wgs84 = None
for layer, style in preview_layers.items():
    gdf = gpd.read_file(output_path, layer=layer).to_crs(epsg=4326)
    last_gdf_wgs84 = gdf
    m.add_gdf(gdf, layer_name=layer, style=style, info_mode="on_click")

if last_gdf_wgs84 is not None:
    m.zoom_to_gdf(last_gdf_wgs84)

m